In [1]:
from oqd_compiler_infrastructure import Post, PrettyPrint

from oqd_core.frontend.atomic import parse_atomic

printer = Post(PrettyPrint())

with open("test.atomic", mode="r", encoding="utf8") as f:
    source = f.read()

circuit = parse_atomic(source)
# print(printer(circuit))

In [2]:
from oqd_core.atomic import AtomicCFGBuilder

printer = Post(PrettyPrint())

with open("test.atomic", mode="r", encoding="utf8") as f:
    source = f.read()

circuit = parse_atomic(source)
cfg = AtomicCFGBuilder()(circuit)

In [3]:
from oqd_core.atomic import (
    AtomicBoundsChecker,
    AtomicCFGBuilder,
    AtomicTypeChecker,
    AvailableVariableAnalysis,
    serialize_atomic,
)

In [4]:
avail_checker = AvailableVariableAnalysis()
avail_res = avail_checker.analyze(cfg)

In [5]:
bounds_checker = AtomicBoundsChecker()
bounds_res = bounds_checker.analyze(cfg)

In [6]:
checker = AtomicTypeChecker()
res = checker.analyze(cfg)

In [7]:
from oqd_core.compiler.atomic import compile_atomic_circuit

circuit, cfg = compile_atomic_circuit(source)

In [8]:
circuit

AtomicCircuit(class_='AtomicCircuit', statements=[Declaration(class_='Declaration', name='ions', value=IonRegister(class_='IonRegister', size=Constant(class_='Constant', value=2))), Declaration(class_='Declaration', name='q0', value=Extract(class_='Extract', value=Access(class_='Access', name='ions'), index=Constant(class_='Constant', value=0))), Declaration(class_='Declaration', name='q1', value=Extract(class_='Extract', value=Access(class_='Access', name='ions'), index=Constant(class_='Constant', value=1))), Declaration(class_='Declaration', name='targets', value=AtomicList(class_='AtomicList', values=[Access(class_='Access', name='q0'), Access(class_='Access', name='q1')])), Declaration(class_='Declaration', name='pi', value=Constant(class_='Constant', value=3.14159)), Declaration(class_='Declaration', name='omega', value=RuntimeVar(class_='RuntimeVar', name='#omega')), Declaration(class_='Declaration', name='phi', value=RuntimeVar(class_='RuntimeVar', name='#phi')), Declaration(cla

In [9]:
from oqd_compiler_infrastructure import cfg_to_dot

dot = cfg_to_dot(cfg, serialize=serialize_atomic, max_lines=5)
print(dot.source)
dot.render("cfg", format="png", cleanup=True)

digraph {
	0 [label=" Block #0
------------------------
--------- tags ---------
__label__: entry"]
	0 -> 1
	1 [label=" Block #1
------------------------
ions = ionreg(2)
q0 = ions[0]
q1 = ions[1]
targets = [q0, q1]
pi = 3.14159
..."]
	1 -> 2
	2 [label="Branch Block #2
------------------------
Condition: flag
--------- tags ---------
__scf__: IfElse"]
	2 -> 3 [label=true]
	2 -> 4 [label=false]
	3 [label=" Block #3
------------------------
pulse(beam_mw, 2e-06, targets, false)"]
	3 -> 5
	4 [label=" Block #4
------------------------
pulse(beam_det, 2e-05, targets, true)"]
	4 -> 5
	5 [label="Branch Block #5
------------------------
Condition: n > 0
--------- tags ---------
__scf__: While"]
	5 -> 6 [label=true]
	5 -> 11 [label=false]
	6 [label=" Block #6
------------------------
parallel {
  pulse(beam_mw, 1e-06, targets, false)
}
n = n - 1"]
	6 -> 7
	7 [label="Branch Block #7
------------------------
Condition: n == 1
--------- tags ---------
__scf__: IfElse"]
	7 -> 8 [label=true]
	7 -> 9

'cfg.png'